In [1]:
from llama_index.llms.ollama import Ollama

from llama_index.core.workflow import Context

In [2]:
import asyncio
from llama_index.core.agent.workflow import FunctionAgent



# Define a simple calculator tool
def multiply(a: float, b: float) -> float:
    """Useful for multiplying two numbers."""
    return a * b


# Create an agent workflow with our calculator tool
agent = FunctionAgent(
    tools=[multiply],
    llm=Ollama(
        model="llama3.2",
        request_timeout=360.0,
        # Manually set the context window to limit memory usage
        context_window=8000,
    ),
    system_prompt="You are a helpful assistant that can multiply two numbers.",
)


In [3]:
ctx = Context(agent)

In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.ollama import OllamaEmbedding
import os

/Users/eddieogola/dev/projects/ain/server/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Settings control global defaults
Settings.embed_model = OllamaEmbedding(
    model_name="embeddinggemma",
    request_timeout=360.0,
)
Settings.llm = Ollama(
    model="llama3.2",
    request_timeout=360.0,
    # Manually set the context window to limit memory usage
    context_window=8000,
)

In [10]:
# Create a RAG tool using LlamaIndex
documents = SimpleDirectoryReader("data").load_data()
print(documents[0].id_)

f310c2c0-d8cc-47fd-8e8f-adbb35c66e65


In [ ]:
# # Indexing
# index = VectorStoreIndex.from_documents(
#     documents,
#     # we can optionally override the embed_model here
#     embed_model=Settings.embed_model,
# )
# # Save the index
# index.storage_context.persist("storage")

In [6]:

# Later, load the index
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="storage")
index = load_index_from_storage(
    storage_context,
    # we can optionally override the embed_model here
    # it's important to use the same embed_model as the one used to build the index
    embed_model=Settings.embed_model,
)

Loading llama_index.core.storage.kvstore.simple_kvstore from storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/index_store.json.


In [14]:
query_engine = index.as_query_engine(streaming=True, similarity_top_k=3)


In [15]:
response = query_engine.query("What is attention in transformers")
response.print_response_stream()

Attention mechanisms allow models to focus on specific parts of input data when generating output. This process helps to identify long-distance dependencies within a sentence, like the connection between the verb "making" and the phrase "more difficult".